In [32]:
import os
import torch
import numpy as np
import pandas as pd
import json
from tqdm import tqdm
import spacy
from spacy.training import offsets_to_biluo_tags
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from keras.preprocessing.sequence import pad_sequences
from pytorch_pretrained_bert import BertTokenizer, BertConfig, BertForTokenClassification

In [33]:
# Load spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

Loading spaCy model...


In [34]:
# Adding '\n' to the default spacy tokenizer
prefixes = ('\n', ) + tuple(nlp.Defaults.prefixes)
prefix_regex = spacy.util.compile_prefix_regex(prefixes)
nlp.tokenizer.prefix_search = prefix_regex.search

In [35]:
# Personal Custom Tags Dictionary (same as used in training)
entity_dict = {
    'Name': 'NAME', 
    'College Name': 'CLG',
    'Degree': 'DEG',
    'Graduation Year': 'GRADYEAR',
    'Years of Experience': 'YOE',
    'Companies worked at': 'COMPANY',
    'Designation': 'DESIG',
    'Skills': 'SKILLS',
    'Location': 'LOC',
    'Email Address': 'EMAIL'
}
print("Entity dictionary:", entity_dict)

Entity dictionary: {'Name': 'NAME', 'College Name': 'CLG', 'Degree': 'DEG', 'Graduation Year': 'GRADYEAR', 'Years of Experience': 'YOE', 'Companies worked at': 'COMPANY', 'Designation': 'DESIG', 'Skills': 'SKILLS', 'Location': 'LOC', 'Email Address': 'EMAIL'}


In [36]:
# Function to merge overlapping intervals (spans) from the annotations
def merge_intervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []
    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [37]:
# Extract entities from annotations
def get_entities(df):
    entities = []
    for i in range(len(df)):
        entity = []
        if df['annotation'][i] is not None:
            for annot in df['annotation'][i]:
                try:
                    ent = entity_dict[annot['label'][0]]
                    start = annot['points'][0]['start']
                    end = annot['points'][0]['end'] + 1
                    entity.append((start, end, ent))
                except:
                    pass
            entity = merge_intervals(entity)
        entities.append(entity)
    return entities

In [38]:
# Convert data to BILUO format for NER evaluation
def get_test_data(df):
    tags = []
    sentences = []
    for i in range(len(df)):
        text = df['content'][i]
        entities = df['entities'][i]
        doc = nlp(text)
        # Convert to BILUO tags (Beginning, Inside, Last, Unit, Outside)
        tag = offsets_to_biluo_tags(doc, entities)
        tmp = pd.DataFrame([list(doc), tag]).T
        # Split by sentences (using periods as delimiters)
        loc = []
        for i in range(len(tmp)):
            if tmp[0][i].text == '.' and tmp[1][i] == 'O':
                loc.append(i)
        loc.append(len(doc))
        last = 0
        data = []
        for pos in loc:
            data.append([list(doc)[last:pos], tag[last:pos]])
            last = pos
        # Only keep sentences with at least one entity
        for d in data:
            tag = ['O' if t == '-' else t for t in d[1]]
            if len(set(tag)) > 1:  # If there's more than just 'O' tags
                sentences.append(d[0])
                tags.append(tag)
    return sentences, tags

In [39]:
# Tokenize sentences and align labels
def get_tokenized_test_data(sentences, tags, tokenizer):
    tokenized_texts = []
    word_piece_labels = []
    original_tokens = []
    for word_list, label in zip(sentences, tags):
        # Add [CLS] at the front
        temp_label = ['[CLS]']
        temp_token = ['[CLS]']
        orig_tokens = ['[CLS]']
        for word, lab in zip(word_list, label):
            orig_tokens.append(word.text)
            token_list = tokenizer.tokenize(word.text)
            for m, token in enumerate(token_list):
                temp_token.append(token)
                if m == 0:
                    temp_label.append(lab)  # First subword gets the original tag
                else:
                    temp_label.append('X')  # Other subwords get 'X' tag
        # Add [SEP] at the end
        temp_label.append('[SEP]')
        temp_token.append('[SEP]')
        orig_tokens.append('[SEP]')
        tokenized_texts.append(temp_token)
        word_piece_labels.append(temp_label)
        original_tokens.append(orig_tokens)
    return tokenized_texts, word_piece_labels, original_tokens

In [44]:
# Main evaluation function
def evaluate_model(model_path, test_file, max_len=128):
    print(f"Loading test data from {test_file}...")
    
    # Check if test file exists
    if not os.path.exists(test_file):
        print(f"Error: Test file {test_file} not found!")
        return
    
    # Load test data
    try:
        df = pd.read_json(test_file)
        print(f"Loaded test dataset with {len(df)} rows")
    except Exception as e:
        print(f"Error loading test data: {e}")
        return
    
    # Check if model exists
    model_file = os.path.join(model_path, "fifth_epoch.pt")
    config_file = os.path.join(model_path, "config.json")
    
    if not os.path.exists(model_file):
        print(f"Error: Model file {model_file} not found!")
        return
    
    # Process test data
    print("Processing test data...")
    df['entities'] = get_entities(df)
    sentences, tags = get_test_data(df)
    print(f"Created {len(sentences)} test sentences")
    
    # Get unique tags
    tag_vals = set(['X', '[CLS]', '[SEP]', 'O'])
    for tag_list in tags:
        tag_vals = tag_vals.union(tag_list)
    
    # Create tag to index mapping
    tag2idx = {t: i for i, t in enumerate(tag_vals)}
    idx2tag = {i: t for t, i in tag2idx.items()}
    
    print(f"Found {len(tag_vals)} unique tags")
    
    # Set up device (GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load tokenizer and model
    print("Loading tokenizer and model...")
    try:
        tokenizer = BertTokenizer.from_pretrained(model_path, do_lower_case=False)
    except:
        print("Could not load tokenizer from model path, using default bert-base-cased")
        tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)
    
    # Load model configuration
    try:
        config = BertConfig.from_json_file(config_file)
    except:
        print("Could not load config.json, creating default config")
        config = BertConfig.from_pretrained('bert-base-cased')
    
    # Load model - FIXED: Pass num_labels directly
    try:
        num_labels = len(tag2idx)
        print(f"Initializing model with {num_labels} labels")
        
        # Try different approaches to initialize the model
        try:
            # Approach 1: Pass num_labels directly
            model = BertForTokenClassification(config, num_labels=num_labels)
        except TypeError:
            try:
                # Approach 2: Use from_pretrained with the model path
                model = BertForTokenClassification.from_pretrained(
                    model_path, 
                    num_labels=num_labels
                )
            except:
                # Approach 3: Initialize with pre-trained weights first
                model = BertForTokenClassification.from_pretrained(
                    'bert-base-cased',
                    num_labels=num_labels
                )
        
        # Load the state dict
        model.load_state_dict(torch.load(model_file, map_location=device))
        model.to(device)
        model.eval()
        print("Model loaded successfully")
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return
    
    # Rest of the function remains the same...
    # Tokenize test data
    print("Tokenizing test data...")
    tokenized_texts, word_piece_labels, original_tokens = get_tokenized_test_data(sentences, tags, tokenizer)
    
    # Convert tokens to IDs and pad sequences
    input_ids = pad_sequences(
        [tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
        maxlen=max_len, dtype="long", truncating="post", padding="post"
    )
    
    # Convert tags to IDs and pad sequences
    tags_ids = pad_sequences(
        [[tag2idx.get(l) for l in lab] for lab in word_piece_labels],
        maxlen=max_len, value=tag2idx["O"], padding="post", 
        dtype="long", truncating="post"
    )
    
    # Create attention masks
    attention_masks = [[float(i > 0) for i in ii] for ii in input_ids]
    
    # Convert to PyTorch tensors
    test_inputs = torch.tensor(input_ids, dtype=torch.long)
    test_tags = torch.tensor(tags_ids, dtype=torch.long)
    test_masks = torch.tensor(attention_masks, dtype=torch.float)
    
    # Create DataLoader
    batch_size = 8
    
    # Run prediction
    print("Running predictions...")
    pred_tags = []
    true_tags = []
    
    # Process in batches
    for i in range(0, len(test_inputs), batch_size):
        batch_inputs = test_inputs[i:i+batch_size].to(device)
        batch_masks = test_masks[i:i+batch_size].to(device)
        batch_tags = test_tags[i:i+batch_size]
        
        # Forward pass
        with torch.no_grad():
            outputs = model(batch_inputs, token_type_ids=None, attention_mask=batch_masks)
            
            # Handle different output formats
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                try:
                    logits = outputs.logits
                except AttributeError:
                    logits = outputs
        # Convert logits to predictions
        logits = logits.detach().cpu().numpy()
        batch_tags = batch_tags.numpy()
        # For each example in the batch
        for j in range(len(logits)):
            example_preds = np.argmax(logits[j], axis=1)
            example_tags = batch_tags[j]
            # Convert indices back to tags, ignoring padding
            length = min(len(tokenized_texts[i+j]), max_len)
            example_pred_tags = [idx2tag[p] for p in example_preds[:length]]
            example_true_tags = [idx2tag[t] for t in example_tags[:length]]
            # Filter out special tokens ([CLS], [SEP], X)
            filtered_preds = []
            filtered_true = []
            for p, t, token in zip(example_pred_tags, example_true_tags, tokenized_texts[i+j][:length]):
                if t not in ['[CLS]', '[SEP]', 'X']:
                    filtered_preds.append(p)
                    filtered_true.append(t)
            pred_tags.extend(filtered_preds)
            true_tags.extend(filtered_true)
    # Calculate metrics
    print("\nEvaluation Results:")
    print("-------------------")
    # Print classification report
    print("\nDetailed Classification Report:")
    print(classification_report(true_tags, pred_tags))
    # Calculate overall metrics
    accuracy = accuracy_score(true_tags, pred_tags)
    precision = precision_score(true_tags, pred_tags, average='weighted')
    recall = recall_score(true_tags, pred_tags, average='weighted')
    f1 = f1_score(true_tags, pred_tags, average='weighted')
    print("\nOverall Metrics:")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    # Calculate metrics per entity type
    print("\nMetrics by Entity Type:")
    entity_types = set([tag for tag in true_tags if tag.startswith('B-') or tag.startswith('U-')])
    for entity in sorted(entity_types):
        entity_name = entity.split('-')[1]  # Extract entity name (e.g., 'NAME' from 'B-NAME')
        # Create binary lists for this entity type
        true_binary = [1 if tag.endswith(entity_name) else 0 for tag in true_tags]
        pred_binary = [1 if tag.endswith(entity_name) else 0 for tag in pred_tags]
        # Calculate metrics
        entity_precision = precision_score(true_binary, pred_binary, average='binary', zero_division=0)
        entity_recall = recall_score(true_binary, pred_binary, average='binary', zero_division=0)
        entity_f1 = f1_score(true_binary, pred_binary, average='binary', zero_division=0)
        print(f"{entity_name}:")
        print(f"  Precision: {entity_precision:.4f}")
        print(f"  Recall:    {entity_recall:.4f}")
        print(f"  F1 Score:  {entity_f1:.4f}")
    print("\nEvaluation complete!")

In [45]:
# Run the evaluation
if __name__ == "__main__":
    model_path = "./resume_parser_model/"  # FIXED: Changed from resume_parser_model_interrupted
    test_file = "./test.json"  # Path to the test data
    evaluate_model(model_path, test_file)

Loading test data from ./test.json...
Loaded test dataset with 106 rows
Processing test data...


C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Vijay Shinde
asst. sales manager in M - S PLANET T..." with entities "[(0, 12, 'NAME'), (13, 32, 'DESIG'), (40, 67, 'COM...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Jaykumar Shah
Sales Manager

Mumbai, Maharashtra -..." with entities "[(0, 13, 'NAME'), (14, 27, 'DESIG'), (29, 35, 'LOC...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserW

Created 547 test sentences
Found 44 unique tags
Using device: cuda
Loading tokenizer and model...
Initializing model with 44 labels
Error loading model: Error(s) in loading state_dict for BertForTokenClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([43, 768]) from checkpoint, the shape in current model is torch.Size([44, 768]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([43]) from checkpoint, the shape in current model is torch.Size([44]).


Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_1192\4061838323.py", line 86, in evaluate_model
    model.load_state_dict(torch.load(model_file, map_location=device))
  File "C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\torch\nn\modules\module.py", line 2581, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for BertForTokenClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([43, 768]) from checkpoint, the shape in current model is torch.Size([44, 768]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([43]) from checkpoint, the shape in current model is torch.Size([44]).
